In [ ]:
from pathlib import Path
import pandas as pd

RAW = Path("data/raw")

def load_folder(folder):
    """Read every review .txt file in a folder (and its subfolders), skipping Readme files."""
    rows = []
    for f in folder.rglob("*.txt"):
        if "readme" in f.name.lower():
            continue
        with open(f, encoding="latin-1") as fh:
            for line in fh:
                line = line.strip()
                if line:
                    rows.append({"product": f.stem, "raw": line})
    return pd.DataFrame(rows)

In [6]:
folders = ["CustomerReviewData", "Reviews-9-products", "CustomerReviews-3-domains"]

df = pd.concat(
    [load_folder(RAW / name).assign(source=name) for name in folders],
    ignore_index=True,
)
print(df.shape)
print(df["source"].value_counts())

(10992, 3)
source
Reviews-9-products           4584
CustomerReviewData           4309
CustomerReviews-3-domains    2099
Name: count, dtype: int64


In [7]:
import re

SCORE = re.compile(r"\[([+-])(\d?)\]")   

def parse_line(line):
    # skip file header, review titles, and lines without a sentence
    if line.startswith("*") or line.startswith("[t]") or "##" not in line:
        return None
    tags, sentence = line.split("##", 1)
    scores = [(1 if sign == "+" else -1) * int(num or 1)
              for sign, num in SCORE.findall(tags)]
    if not scores:          # no opinion on this sentence
        return None
    total = sum(scores)
    if total == 0:          # mixed opinion that cancels out
        return None
    return {"text": sentence.strip().lower(),
            "label": "positive" if total > 0 else "negative"}

parsed = df["raw"].apply(parse_line).dropna()
data = pd.DataFrame(parsed.tolist())
data["product"] = df.loc[parsed.index, "product"].values
data["source"] = df.loc[parsed.index, "source"].values

print(data.shape)
print(data["label"].value_counts())
print(pd.crosstab(data["source"], data["label"]))
data.head(10)

(4579, 4)
label
positive    2979
negative    1600
Name: count, dtype: int64
label                      negative  positive
source                                       
CustomerReviewData              638      1081
CustomerReviews-3-domains       221       549
Reviews-9-products              741      1349


,text,label,product,source
0,im a more happier person after discovering the...,positive,Apex AD2600 Progressive-scan DVD player,CustomerReviewData
1,"but , if you 're looking for my opinion of the...",positive,Apex AD2600 Progressive-scan DVD player,CustomerReviewData
2,it practically plays almost everything you giv...,positive,Apex AD2600 Progressive-scan DVD player,CustomerReviewData
3,i 've had the player for about 2 years now and...,positive,Apex AD2600 Progressive-scan DVD player,CustomerReviewData
4,for the price it is a well spent investment !,positive,Apex AD2600 Progressive-scan DVD player,CustomerReviewData
5,"this is by far the nicest one , in so many ways .",positive,Apex AD2600 Progressive-scan DVD player,CustomerReviewData
6,it 's very sleek looking with a very good fron...,positive,Apex AD2600 Progressive-scan DVD player,CustomerReviewData
7,its fast-forward and rewind work much more smo...,positive,Apex AD2600 Progressive-scan DVD player,CustomerReviewData
8,"it plays alternate video formats ( vcds , svcd...",positive,Apex AD2600 Progressive-scan DVD player,CustomerReviewData
9,and amazon.com has it for such a great price -...,positive,Apex AD2600 Progressive-scan DVD player,CustomerReviewData


In [10]:
before = len(data)
data = data.drop_duplicates(subset="text").reset_index(drop=True)
print(f"removed {before - len(data)} duplicate rows → {len(data)} left")


removed 0 duplicate rows → 4574 left


In [11]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(
    data, test_size=0.2, random_state=42, stratify=data["label"]
)

OUT = Path("data/processed")
OUT.mkdir(parents=True, exist_ok=True)
train.to_csv(OUT / "train.csv", index=False)
test.to_csv(OUT / "test.csv", index=False)

print("train:", train.shape, train["label"].value_counts(normalize=True).round(2).to_dict())
print("test: ", test.shape,  test["label"].value_counts(normalize=True).round(2).to_dict())

train: (3659, 4) {'positive': 0.65, 'negative': 0.35}
test:  (915, 4) {'positive': 0.65, 'negative': 0.35}
